# Notebook 01 — Data Engineering
**Student 1 · Data Engineer**

This notebook downloads historical price and volume data for 27 NIFTY 50 constituents
from Yahoo Finance, performs quality checks, aligns to the NIFTY index trading calendar,
computes daily returns, and saves all processed artefacts to `data/`.

**Run order:** This notebook must be run *first*.

In [ ]:
# ── Cell 1 · Imports ──────────────────────────────────────────────────────────
import os
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf

warnings.filterwarnings('ignore')
logging.getLogger('yfinance').setLevel(logging.ERROR)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded successfully.')

In [ ]:
# ── Cell 2 · Configuration ────────────────────────────────────────────────────
TICKERS = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
    'HINDUNILVR.NS', 'ITC.NS', 'SUNPHARMA.NS', 'LT.NS', 'MARUTI.NS',
    'BHARTIARTL.NS', 'TITAN.NS', 'NESTLEIND.NS', 'AXISBANK.NS', 'KOTAKBANK.NS',
    'BAJFINANCE.NS', 'WIPRO.NS', 'HCLTECH.NS', 'ULTRACEMCO.NS', 'ASIANPAINT.NS',
    'TECHM.NS', 'POWERGRID.NS', 'NTPC.NS', 'ONGC.NS', 'COALINDIA.NS',
    'BAJAJFINSV.NS', 'DRREDDY.NS'
]

UNIVERSE_INFO = {
    'RELIANCE.NS':   ('Reliance Industries',    'Energy'),
    'TCS.NS':        ('Tata Consultancy Services','IT'),
    'HDFCBANK.NS':   ('HDFC Bank',              'Banking'),
    'INFY.NS':       ('Infosys',                'IT'),
    'ICICIBANK.NS':  ('ICICI Bank',             'Banking'),
    'HINDUNILVR.NS': ('Hindustan Unilever',     'FMCG'),
    'ITC.NS':        ('ITC',                    'FMCG'),
    'SUNPHARMA.NS':  ('Sun Pharma',             'Pharma'),
    'LT.NS':         ('Larsen & Toubro',        'Industrials'),
    'MARUTI.NS':     ('Maruti Suzuki',          'Auto'),
    'BHARTIARTL.NS': ('Bharti Airtel',          'Telecom'),
    'TITAN.NS':      ('Titan',                  'Consumer'),
    'NESTLEIND.NS':  ('Nestlé India',           'FMCG'),
    'AXISBANK.NS':   ('Axis Bank',              'Banking'),
    'KOTAKBANK.NS':  ('Kotak Mahindra Bank',    'Banking'),
    'BAJFINANCE.NS': ('Bajaj Finance',          'Banking'),
    'WIPRO.NS':      ('Wipro',                  'IT'),
    'HCLTECH.NS':    ('HCL Technologies',       'IT'),
    'ULTRACEMCO.NS': ('UltraTech Cement',       'Industrials'),
    'ASIANPAINT.NS': ('Asian Paints',           'Consumer'),
    'TECHM.NS':      ('Tech Mahindra',          'IT'),
    'POWERGRID.NS':  ('Power Grid',             'Energy'),
    'NTPC.NS':       ('NTPC',                   'Energy'),
    'ONGC.NS':       ('ONGC',                   'Energy'),
    'COALINDIA.NS':  ('Coal India',             'Energy'),
    'BAJAJFINSV.NS': ('Bajaj Finserv',          'Banking'),
    'DRREDDY.NS':    ("Dr. Reddy's",            'Pharma'),
}

BENCHMARK  = '^NSEI'
START_DATE = '2015-01-01'
END_DATE   = '2026-11-01'
RANDOM_SEED = 42

DATA_DIR    = '../data'
RAW_DIR     = os.path.join(DATA_DIR, 'raw')
PROC_DIR    = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = '../results'

for d in [RAW_DIR, PROC_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Universe : {len(TICKERS)} stocks')
print(f'Period   : {START_DATE} → {END_DATE}')
print(f'Benchmark: {BENCHMARK}')

In [ ]:
# ── Cell 3 · Download Data ────────────────────────────────────────────────────
def download_ticker(ticker, start, end):
    """Download OHLCV for a single ticker; return None on failure."""
    try:
        df = yf.download(ticker, start=start, end=end,
                         auto_adjust=True, progress=False)
        if df.empty or len(df) < 100:
            return None
        df.index = pd.to_datetime(df.index)
        df.index.name = 'date'
        return df
    except Exception as e:
        print(f'  ERROR downloading {ticker}: {e}')
        return None

print('Downloading benchmark index …')
nifty_raw = download_ticker(BENCHMARK, START_DATE, END_DATE)
print(f'  NIFTY rows: {len(nifty_raw) if nifty_raw is not None else 0}')

print('\nDownloading stock universe …')
raw_data   = {}
failed     = []
for tkr in TICKERS:
    df = download_ticker(tkr, START_DATE, END_DATE)
    if df is not None:
        raw_data[tkr] = df
        print(f'  ✓ {tkr:20s}  rows={len(df)}')
    else:
        failed.append(tkr)
        print(f'  ✗ {tkr:20s}  FAILED / insufficient data')

print(f'\nSuccessfully downloaded: {len(raw_data)}/{len(TICKERS)}')
if failed:
    print(f'Failed tickers: {failed}')

In [ ]:
# ── Cell 4 · Quality Checks ───────────────────────────────────────────────────
print('=' * 65)
print(f'  {"Ticker":<20} {"Rows":>6} {"NaN%":>6} {"Gaps":>6} {"Dups":>5} {"ZeroP":>6}')
print('=' * 65)

quality_report = []
tickers_to_drop = []

for tkr, df in raw_data.items():
    close = df['Close'].squeeze()
    nan_pct  = close.isna().mean() * 100
    # Detect gaps > 5 trading days
    idx_diff = pd.Series(close.index).diff().dt.days
    gaps     = int((idx_diff > 5).sum())
    dups     = int(close.index.duplicated().sum())
    zero_neg = int((close <= 0).sum())

    quality_report.append({
        'ticker': tkr, 'rows': len(df),
        'nan_pct': round(nan_pct, 2), 'gaps': gaps,
        'duplicates': dups, 'zero_neg_prices': zero_neg
    })
    print(f'  {tkr:<20} {len(df):>6} {nan_pct:>6.2f} {gaps:>6} {dups:>5} {zero_neg:>6}')

    # Drop if too many problems
    if nan_pct > 20 or zero_neg > 0:
        tickers_to_drop.append(tkr)

print('=' * 65)
quality_df = pd.DataFrame(quality_report)

if tickers_to_drop:
    print(f'\nDropping problematic tickers: {tickers_to_drop}')
    for t in tickers_to_drop:
        raw_data.pop(t, None)
else:
    print('\nAll tickers passed quality checks.')

ACTIVE_TICKERS = list(raw_data.keys())
print(f'Active universe: {len(ACTIVE_TICKERS)} stocks')

In [ ]:
# ── Cell 5 · Alignment to NIFTY Trading Calendar ─────────────────────────────
# Use NIFTY index dates as the reference calendar
nifty_close = nifty_raw['Close'].squeeze()
nifty_dates = nifty_close.index

# Build wide price and volume DataFrames, then reindex + forward-fill (max 3 days)
price_dict  = {}
volume_dict = {}

for tkr, df in raw_data.items():
    close  = df['Close'].squeeze()
    volume = df['Volume'].squeeze() if 'Volume' in df.columns else pd.Series(
                dtype=float, name=tkr)
    # Reindex to NIFTY dates, forward-fill up to 3 consecutive missing days
    close_aligned  = close.reindex(nifty_dates).ffill(limit=3)
    volume_aligned = volume.reindex(nifty_dates).ffill(limit=3)
    price_dict[tkr]  = close_aligned
    volume_dict[tkr] = volume_aligned

prices_df  = pd.DataFrame(price_dict)
volumes_df = pd.DataFrame(volume_dict)
nifty_aligned = nifty_close.reindex(nifty_dates)

# Drop rows where more than 50% of stocks have NaN (early data gaps)
threshold = 0.5 * len(ACTIVE_TICKERS)
prices_df = prices_df[prices_df.notna().sum(axis=1) >= threshold]
volumes_df = volumes_df.reindex(prices_df.index)
nifty_aligned = nifty_aligned.reindex(prices_df.index)

print(f'Aligned price matrix : {prices_df.shape}')
print(f'Date range           : {prices_df.index[0].date()} → {prices_df.index[-1].date()}')
print(f'Remaining NaN in prices: {prices_df.isna().sum().sum()}')

In [ ]:
# ── Cell 6 · Returns Computation ─────────────────────────────────────────────
returns_df     = prices_df.pct_change()
nifty_returns  = nifty_aligned.pct_change()

# Drop the first row (NaN from pct_change)
returns_df    = returns_df.iloc[1:]
nifty_returns = nifty_returns.iloc[1:]

# Clip extreme returns (> |50%| per day → data errors)
returns_df = returns_df.clip(lower=-0.5, upper=0.5)

print('Daily Returns Summary')
print(returns_df.describe().round(4))

# Save returns
returns_df.to_csv(os.path.join(PROC_DIR, 'returns.csv'))
print(f'\nSaved returns.csv  shape={returns_df.shape}')

In [ ]:
# ── Cell 7 · Universe CSV ─────────────────────────────────────────────────────
universe_rows = []
for tkr in ACTIVE_TICKERS:
    name, sector = UNIVERSE_INFO.get(tkr, (tkr, 'Unknown'))
    universe_rows.append({'ticker': tkr, 'name': name, 'sector': sector})

universe_df = pd.DataFrame(universe_rows)
universe_df.to_csv(os.path.join(RAW_DIR, 'universe.csv'), index=False)
print('Universe:')
print(universe_df.to_string(index=False))

In [ ]:
# ── Cell 8 · Normalised Price Plot ────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Panel 1: All stocks normalised to 100
ax = axes[0]
base = prices_df.iloc[0]
normalised = prices_df.div(base) * 100

sector_colors = {
    'IT': '#1f77b4', 'Banking': '#ff7f0e', 'Energy': '#2ca02c',
    'FMCG': '#d62728', 'Pharma': '#9467bd', 'Industrials': '#8c564b',
    'Auto': '#e377c2', 'Telecom': '#7f7f7f', 'Consumer': '#bcbd22'
}

for tkr in ACTIVE_TICKERS:
    sector = UNIVERSE_INFO.get(tkr, ('', 'Unknown'))[1]
    color  = sector_colors.get(sector, '#17becf')
    ax.plot(normalised.index, normalised[tkr], alpha=0.6, lw=1, color=color)

# NIFTY overlay
nifty_norm = nifty_aligned.reindex(prices_df.index)
nifty_norm = (nifty_norm / nifty_norm.iloc[0]) * 100
ax.plot(nifty_norm.index, nifty_norm, color='black', lw=2.5, label='NIFTY 50')

# Legend for sectors
from matplotlib.lines import Line2D
legend_handles = [Line2D([0], [0], color=c, lw=2, label=s)
                  for s, c in sector_colors.items()]
legend_handles.append(Line2D([0], [0], color='black', lw=2.5, label='NIFTY 50'))
ax.legend(handles=legend_handles, loc='upper left', fontsize=8, ncol=2)
ax.set_title('Normalised Prices (Base = 100 at start)', fontsize=13)
ax.set_ylabel('Index (100 = start)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Panel 2: Sector median
ax2 = axes[1]
sectors = sorted(set(UNIVERSE_INFO[t][1] for t in ACTIVE_TICKERS if t in UNIVERSE_INFO))
for sec in sectors:
    tks = [t for t in ACTIVE_TICKERS if UNIVERSE_INFO.get(t, ('',''))[1] == sec]
    if tks:
        sec_norm = normalised[tks].median(axis=1)
        color    = sector_colors.get(sec, '#17becf')
        ax2.plot(sec_norm.index, sec_norm, lw=2, label=sec, color=color)

ax2.plot(nifty_norm.index, nifty_norm, color='black', lw=2.5, label='NIFTY 50', ls='--')
ax2.legend(loc='upper left', fontsize=8, ncol=3)
ax2.set_title('Sector Median Normalised Price', fontsize=13)
ax2.set_ylabel('Index (100 = start)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'normalised_prices.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {fig_path}')

In [ ]:
# ── Cell 9 · Data Summary Table ───────────────────────────────────────────────
summary_rows = []
for tkr in ACTIVE_TICKERS:
    col    = prices_df[tkr]
    valid  = col.dropna()
    name, sector = UNIVERSE_INFO.get(tkr, (tkr, 'Unknown'))
    summary_rows.append({
        'ticker': tkr, 'name': name, 'sector': sector,
        'start': valid.index[0].date() if len(valid) > 0 else None,
        'end':   valid.index[-1].date() if len(valid) > 0 else None,
        'total_rows': len(col),
        'valid_rows': len(valid),
        'nan_count':  col.isna().sum(),
        'first_price': round(valid.iloc[0], 2) if len(valid) > 0 else None,
        'last_price':  round(valid.iloc[-1], 2) if len(valid) > 0 else None,
    })

summary_df = pd.DataFrame(summary_rows)
print('\n=== Data Quality Summary ===')
display_cols = ['ticker', 'sector', 'start', 'end', 'valid_rows', 'nan_count']
print(summary_df[display_cols].to_string(index=False))
print(f'\nDate range  : {prices_df.index[0].date()} → {prices_df.index[-1].date()}')
print(f'Trading days: {len(prices_df)}')
print(f'Stocks      : {len(ACTIVE_TICKERS)}')
if failed:
    print(f'Dropped (download fail): {failed}')
if tickers_to_drop:
    print(f'Dropped (quality fail) : {tickers_to_drop}')

In [ ]:
# ── Cell 10 · Save All Outputs ────────────────────────────────────────────────
# Prices
prices_df.to_csv(os.path.join(RAW_DIR, 'prices.csv'))
print(f'Saved prices.csv        {prices_df.shape}')

# Volume
volumes_df.to_csv(os.path.join(RAW_DIR, 'volumes.csv'))
print(f'Saved volumes.csv       {volumes_df.shape}')

# NIFTY index
nifty_aligned_df = nifty_aligned.to_frame(name='Close')
nifty_aligned_df['Return'] = nifty_returns.reindex(nifty_aligned_df.index)
nifty_aligned_df.to_csv(os.path.join(RAW_DIR, 'index.csv'))
print(f'Saved index.csv         {nifty_aligned_df.shape}')

# Returns (also save to processed)
returns_df.to_csv(os.path.join(PROC_DIR, 'returns.csv'))
print(f'Saved returns.csv       {returns_df.shape}')

# Universe
universe_df.to_csv(os.path.join(RAW_DIR, 'universe.csv'), index=False)
print(f'Saved universe.csv      {universe_df.shape}')

print('\n✅  Notebook 01 complete — all data artefacts saved.')